# Submission 08 - Identity Target + Frequency

This submission uses the winning Experiment 23B approach.

It keeps the original numerical features and categorical features, adds leakage-safe out-of-fold exact-value target encodings for numerical identities, and adds exact-value frequency features.

No pair-identity features are included because Experiment 23B slightly outperformed Experiment 23C.

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

PROJECT_ROOT = Path(r'C:\Users\aakif\Documents\DataCompetition')
TRAIN_PATH = PROJECT_ROOT / 'data' / 'train.csv'
TEST_PATH = PROJECT_ROOT / 'data' / 'test.csv'
SAMPLE_PATH = PROJECT_ROOT / 'data' / 'sample_submission.csv'
OUTPUT_PATH = PROJECT_ROOT / 'submissions' / 'submission_08.csv'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

X_train = train.drop(columns=['Will_Buy_EV', 'id']).copy()
y = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1}).astype(int)
X_test = test.drop(columns=['id']).copy()

numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=['number']).columns.tolist()

print('Training rows:', len(X_train))
print('Test rows:', len(X_test))
print('Numeric columns:', numeric_cols)
print('Categorical columns:', categorical_cols)

Training rows: 668665
Test rows: 286571
Numeric columns: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
Categorical columns: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


In [2]:
def make_identity_key(series):
    return series.astype('string').fillna('__MISSING__')


def fit_mapping(values, target, smoothing=20):
    temp = pd.DataFrame({
        'value': values,
        'target': target.to_numpy()
    })

    global_mean = float(target.mean())
    stats = temp.groupby('value', dropna=False)['target'].agg(['mean', 'count'])

    smoothed = (
        stats['count'] * stats['mean'] + smoothing * global_mean
    ) / (stats['count'] + smoothing)

    return smoothed.to_dict(), global_mean


def apply_mapping(values, mapping, global_mean):
    return values.map(mapping).fillna(global_mean).astype(float)


def add_oof_identity_features(X_fit, y_fit, X_apply, columns, n_splits=3, smoothing=20):
    X_fit = X_fit.copy()
    X_apply = X_apply.copy()

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    for col in columns:
        fit_keys = make_identity_key(X_fit[col])
        apply_keys = make_identity_key(X_apply[col])

        oof_values = np.zeros(len(X_fit), dtype=float)

        for train_idx, fold_idx in skf.split(X_fit, y_fit):
            fold_keys = fit_keys.iloc[train_idx]
            fold_target = y_fit.iloc[train_idx]

            mapping, global_mean = fit_mapping(
                fold_keys,
                fold_target,
                smoothing=smoothing
            )

            oof_values[fold_idx] = apply_mapping(
                fit_keys.iloc[fold_idx],
                mapping,
                global_mean
            ).to_numpy()

        full_mapping, full_global_mean = fit_mapping(
            fit_keys,
            y_fit,
            smoothing=smoothing
        )

        X_fit[f'{col}__identity_target'] = oof_values
        X_apply[f'{col}__identity_target'] = apply_mapping(
            apply_keys,
            full_mapping,
            full_global_mean
        ).to_numpy()

        frequencies = fit_keys.value_counts(dropna=False)

        X_fit[f'{col}__identity_frequency'] = fit_keys.map(
            frequencies
        ).fillna(0).astype(float).to_numpy()

        X_apply[f'{col}__identity_frequency'] = apply_keys.map(
            frequencies
        ).fillna(0).astype(float).to_numpy()

    return X_fit, X_apply

In [3]:
print('Building Experiment 23B features...')

X_train_encoded, X_test_encoded = add_oof_identity_features(
    X_train,
    y,
    X_test,
    numeric_cols,
    n_splits=3,
    smoothing=20
)

print('Original feature count:', X_train.shape[1])
print('Final feature count:', X_train_encoded.shape[1])

print('Added identity target features:', len(numeric_cols))
print('Added identity frequency features:', len(numeric_cols))

Building Experiment 23B features...
Original feature count: 13
Final feature count: 27
Added identity target features: 7
Added identity frequency features: 7


In [4]:
numeric_features = X_train_encoded.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train_encoded.select_dtypes(exclude=['number']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

model = XGBClassifier(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.04,
    min_child_weight=2,
    subsample=0.90,
    colsample_bytree=0.85,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

print('Training Submission 08 model...')
pipeline.fit(X_train_encoded, y)

test_predictions = pipeline.predict_proba(X_test_encoded)[:, 1]

print('Prediction count:', len(test_predictions))
print('Prediction min:', float(test_predictions.min()))
print('Prediction max:', float(test_predictions.max()))
print('Prediction mean:', float(test_predictions.mean()))

Training Submission 08 model...
Prediction count: 286571
Prediction min: 3.3875542158057215e-06
Prediction max: 0.9998179078102112
Prediction mean: 0.17479024827480316


In [5]:
submission = sample_submission.copy()

if len(submission) != len(test_predictions):
    raise ValueError(
        f'Sample submission has {len(submission)} rows, '
        f'but predictions have {len(test_predictions)} rows.'
    )

submission['Will_Buy_EV'] = test_predictions

if submission['Will_Buy_EV'].isna().any():
    raise ValueError('Submission contains NaN predictions.')

if not np.isfinite(submission['Will_Buy_EV']).all():
    raise ValueError('Submission contains non-finite predictions.')

if not submission['Will_Buy_EV'].between(0, 1).all():
    raise ValueError('Submission predictions must be between 0 and 1.')

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

print('')
print('============================================================')
print('SUBMISSION 08 READY')
print('============================================================')
print(f'Output: {OUTPUT_PATH}')
print(f'Rows: {len(submission)}')
print(f'Columns: {submission.columns.tolist()}')
print('Validation checks: PASSED')
print('')
print('Experiment 23B local ROC-AUC: 0.945243')
print('Current Kaggle benchmark: 0.94450')


SUBMISSION 08 READY
Output: C:\Users\aakif\Documents\DataCompetition\submissions\submission_08.csv
Rows: 286571
Columns: ['id', 'Will_Buy_EV']
Validation checks: PASSED

Experiment 23B local ROC-AUC: 0.945243
Current Kaggle benchmark: 0.94450
